# Implementation of sleep scoring from Watson et al., Neuron (2016)
The paper uses three data sources to classify arousal stages into Awake/NREM/REM/Microarousal, and we want to attempt a further classification of Awake into Active and Quite Wakefulness.

Three data sources are:
- Broadband LFP
- Narrowband theta
- Intracranial EMG

All analyses were originally carried out on ephys recordings from neocortex, downsampled to 1250Hz.

**Broadband LFP** (Low-frequency spectral features)
1. Short-time FFT: 10s window, 1s step, evaluated at log-spaced frequencies between 1–100 Hz
2. z-score
3. PCA
4. Pick PC1. If negative loading on <20Hz frequencies, flip sign.
5. Pick trough in bimodal distribution as threshold between NREM and Wake/REM.

**Narrowband theta**
- Ratio of 5-10Hz/2-16Hz power. Hippocampal channels used where present.

**Intracranial EMG**
1. Bandpass (Butterworth, 4th order) filter in 300-600Hz range
2. Subselect pairs of random, good channels ≥ 2 shanks apart
3. Take mean of Pearson's R in 500ms windows --> **this is the EMG score**

### Imports

In [ ]:
import os
import yaml
import json
import hashlib
import warnings

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
from diptest import diptest
from probeinterface import read_probeinterface
from probeinterface.plotting import plot_probe
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

from sleep_sandbox.io import find_amplifier_files, remote_available, get_or_build, load_preprocessed

### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

raw_data_dir = Path(os.getenv("REMOTE_DATA_DIR"))
session_dir = raw_data_dir.parent          # datetime dir holding probe config + NeuropixelsV2/
derivatives_dir = Path(os.getenv("DERIVATIVES_DIR"))
config_path = os.path.join(repo_root, "config/preprocessing.yml")

# Recording selection: which probe + which AmplifierData _N segment(s) to concatenate.
probe_name = "ProbeB"
suffixes = None                              # int / range / list of AmplifierData _N segments

try:
    amplifier_paths, probe_config_path = find_amplifier_files(session_dir, probe_name, suffixes)
except FileNotFoundError as e:
    probe_config_path = Path(os.getenv("PROBE_CONFIG_PATH"))

# Concatenation is only valid for a contiguous run of segments (no gaps).
seg_ns = sorted(int(p.stem.split("_")[-1]) for p in amplifier_paths)
assert seg_ns == list(range(seg_ns[0], seg_ns[-1] + 1)), f"non-contiguous segments: {seg_ns}"
seg_source = f"{probe_name}_seg{seg_ns[0]}-{seg_ns[-1]}"

# Load preprocessing config
with open(Path(config_path)) as f:
    config = yaml.safe_load(f)

sample_rate = config['recording']['sample_rate']
n_channels = config['recording']['n_channels']
dtype = config['recording']['dtype']
freq_min = config['lfp']['bandpass_filter']['freq_min']
freq_max = config['lfp']['bandpass_filter']['freq_max']
freq_min_highpass = config['preprocessing']['bandpass_filter']['freq_min']
freq_max_highpass = config['preprocessing']['bandpass_filter']['freq_max']
reference = config['lfp']['common_reference']['reference']
operator = config['lfp']['common_reference']['operator']
resample_rate = config['lfp']['resample_rate']

In [ ]:
# Concatenate the selected segments into one continuous recording (bridges internal boundaries).
segments = [se.read_binary(p, sample_rate, dtype, n_channels) for p in amplifier_paths]
recording = si.concatenate_recordings(segments)

### Add probe

In [ ]:
probe_group = read_probeinterface(probe_config_path)
probe = probe_group.probes[0]

active_mask = probe.device_channel_indices != -1
active_probe = probe.get_slice(active_mask)

# active_probe has 384 contacts, all mapped 1:1 to recording channels.
# The full probe has 5120 contacts (384 active); copy_metadata in preprocessors silently
# drops it when contact count != channel count, causing the "no Probe attached" warning.
recording_with_probe = recording.set_probe(active_probe)

### Preprocess

We want to filter this in two ways for a first pass:
- Broadband LFP, roughly 0.5-300 Hz
- Intracranial EMG band, 300-600 Hz

In [ ]:
# Preprocessing pipeline (lazy chain - no data is loaded yet)
# margin_ms must be long enough for the 0.25 Hz high-pass corner (settles over seconds).
margin_ms = 12000

# LFP band ± CMR
recording_filtered_lfp = sp.bandpass_filter(recording_with_probe, freq_min=freq_min, freq_max=freq_max, margin_ms=margin_ms)
recording_lfp_no_cmr = sp.resample(recording_filtered_lfp, resample_rate=resample_rate)
recording_cmr = sp.common_reference(recording_filtered_lfp, reference=reference, operator=operator)
recording_lfp = sp.resample(recording_cmr, resample_rate=resample_rate)

# EMG band
recording_filtered_emg = sp.bandpass_filter(recording_with_probe, freq_min=300, freq_max=600, margin_ms=margin_ms)
recording_emg = sp.resample(recording_filtered_emg, resample_rate=resample_rate)

In [ ]:

preprocessed_dir = Path(derivatives_dir) / "preprocessed"

# The concatenate step pins the exact ordered input set into the params hash + provenance.
concat_step = ("concatenate", {"inputs": [p.stem for p in amplifier_paths]})
lfp_nocmr_steps = [
    concat_step,
    ("bandpass_filter", {"freq_min": freq_min, "freq_max": freq_max, "margin_ms": margin_ms}),
    ("resample", {"resample_rate": resample_rate}),
]
lfp_cmr_steps = [
    concat_step,
    ("bandpass_filter", {"freq_min": freq_min, "freq_max": freq_max, "margin_ms": margin_ms}),
    ("common_reference", {"reference": reference, "operator": operator}),
    ("resample", {"resample_rate": resample_rate}),
]
emg_steps = [
    concat_step,
    ("bandpass_filter", {"freq_min": 300, "freq_max": 600, "margin_ms": margin_ms}),
    ("resample", {"resample_rate": resample_rate}),
]

# Each call loads from disk if the param-matched folder exists, else builds + saves it.
# steps must mirror the actual chain passed in.
#recording_lfp = get_or_build(recording_lfp, seg_source, lfp_cmr_steps, preprocessed_dir, "lfp", n_jobs=20, chunk_duration="30s")
recording_lfp_no_cmr = get_or_build(recording_lfp_no_cmr, seg_source, lfp_nocmr_steps, preprocessed_dir, "lfp", n_jobs=4, chunk_duration="30s")
recording_emg = get_or_build(recording_emg, seg_source, emg_steps, preprocessed_dir, "emg", n_jobs=6, chunk_duration="30s")

For high narrowband theta power, we pick channel 73 (shank 3).

### Dominant spectral feature in broadband LFP

In [ ]:
def broadband_pc1(trace, fs, pc=1):
    """Steps 1-4 on one channel: STFT -> log-freq power -> log10 -> per-freq z-score -> PCA -> oriented PC1."""
    pc_index = pc - 1
    nperseg = int(10 * fs)            # 10 s window
    noverlap = nperseg - int(1 * fs)  # 1 s step
    f, times, spec_lin = signal.spectrogram(trace, fs=fs, window="hann", nperseg=nperseg, noverlap=noverlap)

    freqs = np.logspace(0, 2, 100)  # 1-100 Hz, log-spaced
    spec = np.stack([np.interp(freqs, f, spec_lin[:, i]) for i in range(spec_lin.shape[1])], axis=1)

    logspec = np.log10(spec)
    zspec = (logspec - logspec.mean(axis=1, keepdims=True)) / logspec.std(axis=1, keepdims=True)

    pca = PCA()
    scores = pca.fit_transform(zspec.T)
    pc1, loading = scores[:, pc_index], pca.components_[pc_index]
    if loading[freqs < 20].mean() < 0:  # orient: high score = more slow wave
        pc1, loading = -pc1, -loading
    return pc1, loading, spec, freqs, times


def select_channel_by_dip(rec, fs, stride=16, manual_channel=None):
    """Scan every Nth channel, score PC1 bimodality with Hartigan's dip; return best channel + dip stats."""
    n_ch = rec.get_num_channels()
    candidate_idx = np.arange(0, n_ch, stride)
    dip_stats = np.full(n_ch, np.nan)
    for ci in candidate_idx:
        trace = rec.get_traces(channel_ids=[rec.channel_ids[ci]]).squeeze()
        dip_stats[ci] = diptest(broadband_pc1(trace, fs)[0])[0]
    best_channel = manual_channel if manual_channel is not None else int(np.nanargmax(dip_stats))
    return best_channel, dip_stats, candidate_idx


def pc1_threshold(pc1):
    """Step 5: threshold between the two PC1 modes at the 2-component GMM posterior crossover."""
    gmm = GaussianMixture(n_components=2, random_state=0).fit(pc1.reshape(-1, 1))
    hi_comp = np.argsort(gmm.means_.ravel())[1]            # higher-mean component = NREM
    lo_mean, hi_mean = np.sort(gmm.means_.ravel())
    grid = np.linspace(lo_mean, hi_mean, 1000)
    post_hi = gmm.predict_proba(grid.reshape(-1, 1))[:, hi_comp]
    return grid[np.argmin(np.abs(post_hi - 0.5))]          # posterior crossover (decision boundary)

In [ ]:
# Channel selection: scan every Nth channel, score PC1 bimodality with Hartigan's dip test.
channel_stride = 16     # scan every Nth channel
manual_channel = None   # set to an int channel index to override the dip-test choice

rec = recording_lfp
fs = rec.get_sampling_frequency()

best_channel, dip_stats, candidate_idx = select_channel_by_dip(rec, fs, channel_stride, manual_channel)
print(f"Selected channel {best_channel} (dip={dip_stats[best_channel]:.4f})")

In [ ]:
# Bimodality (Hartigan dip) across scanned channels
plt.figure(figsize=(10, 3))
plt.plot(candidate_idx, dip_stats[candidate_idx], marker=".")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.xlabel("channel index")
plt.ylabel("dip statistic")
plt.legend()

In [ ]:
# Run the broadband slow-wave pipeline on the selected channel.
sw_channel = rec.channel_ids[best_channel]
sw_trace = rec.get_traces(channel_ids=[sw_channel]).squeeze()
sw_pc1, sw_pc1_loading, sw_spec, sw_freqs, sw_times = broadband_pc1(sw_trace, fs)

In [ ]:
# Plot spectrogram of the selected channel
plt.figure(figsize=(10, 4))
plt.pcolormesh(sw_times, sw_freqs, np.log10(sw_spec), shading="auto", cmap="viridis")
plt.yscale("log")
plt.colorbar(label="Power")

In [ ]:
# Plot PC1 (broadbandSlowWave score) over time
plt.figure(figsize=(10, 3))
plt.plot(sw_times, pc1, lw=0.5)
plt.xlabel("Time (s)")
plt.ylabel("PC1 (slow-wave score)")

In [ ]:
# Step 5: threshold between the two modes via a 2-component Gaussian mixture.
sw_threshold = pc1_threshold(sw_pc1)
sw_nrem = sw_pc1 > sw_threshold  # NREM candidate per time bin

In [ ]:
# Plot PC1 distribution (histogram + KDE) with the fitted GMM components
plt.figure(figsize=(6, 4))
plt.hist(sw_pc1, bins=50, density=True, alpha=0.4)

x = np.linspace(sw_pc1.min(), sw_pc1.max(), 1000)
plt.plot(x, gaussian_kde(sw_pc1)(x), color="k", lw=1, label="KDE")
for i, (w, m, s) in enumerate(zip(gmm.weights_, gmm.means_.ravel(), np.sqrt(gmm.covariances_.ravel()))):
    plt.plot(x, w * np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi)), label=f"component {i}")

plt.axvline(sw_threshold, color="r", ls="--", label=f"threshold = {sw_threshold:.2f}")
plt.xlabel("PC1 (slow-wave score)")
plt.ylabel("density")
plt.legend()

### Check classification agreement between channels

Check correlation of binary classifications of sleep state between "best" channel (highest dip test score) and all other channels. Classifications are done by binarising > vs ≤ the threshold computed from a GMM fit with two Gaussians.

In [ ]:
# For each scanned channel: broadband PC1 -> GMM posterior-crossover threshold -> binary NREM series.
# Correlate each channel's binary series against the best channel's.

def pc1_threshold(pc1):
    gmm = GaussianMixture(n_components=2, random_state=0).fit(pc1.reshape(-1, 1))
    hi_comp = np.argsort(gmm.means_.ravel())[1]
    lo_mean, hi_mean = np.sort(gmm.means_.ravel())
    grid = np.linspace(lo_mean, hi_mean, 1000)
    post_hi = gmm.predict_proba(grid.reshape(-1, 1))[:, hi_comp]
    return grid[np.argmin(np.abs(post_hi - 0.5))]

binary_by_channel = {}
for ci in candidate_idx:
    pc1 = broadband_pc1(rec.get_traces(channel_ids=[rec.channel_ids[ci]]).squeeze(), fs)[0]
    binary_by_channel[ci] = (pc1 > pc1_threshold(pc1)).astype(int)

ref_binary = binary_by_channel[best_channel]
corr_by_channel = {ci: np.corrcoef(b, ref_binary)[0, 1] for ci, b in binary_by_channel.items()}

In [ ]:
# Correlation vs channel, coloured by diptest quintile.
ch = np.array(sorted(corr_by_channel))
try:
    corr = np.array([corr_by_channel[c] for c in ch])
except:
    corr = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_binary-sleep-class_corr-with-best-channel_full-probe_channel-indexed.npy")
dip_stats = np.load("/Users/willreith/Documents/phd/aeon/sleep_sandbox/notebooks/abcGolden01_hartigans-diptest_full-probe_channel-indexed.npy")
dips = dip_stats[ch]
quintile = np.digitize(dips, np.quantile(dips, [0.2, 0.4, 0.6, 0.8]))

plt.figure(figsize=(10, 3))
sc = plt.scatter(ch, corr, c=quintile, cmap="viridis")
plt.axvline(best_channel, color="r", ls="--", label=f"best = {best_channel}")
plt.colorbar(sc, label="diptest quintile", ticks=range(5))
plt.xlabel("channel index")
plt.ylabel("corr with best-channel binary")
plt.legend()

### Narrowband theta

### Intracranial EMG